FINAL PRACTICAL WORK WITH POS-TAGGING

In [153]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import spacy
import re
import matplotlib.pyplot as matplot
import pandas

In [154]:
bbc_data = pandas.read_csv("bbc-news.csv")

In [155]:
bbc_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1000 non-null   int64 
 1   index        1000 non-null   int64 
 2   title        1000 non-null   object
 3   pubDate      1000 non-null   object
 4   guid         1000 non-null   object
 5   link         1000 non-null   object
 6   description  1000 non-null   object
dtypes: int64(2), object(5)
memory usage: 54.8+ KB


In [156]:
bbc_data.head(2)

,Unnamed: 0,index,title,pubDate,guid,link,description
0,0,6684,Can I refuse to work?,"Wed, 10 Aug 2022 15:46:18 GMT",https://www.bbc.co.uk/news/business-62147992,https://www.bbc.co.uk/news/business-62147992?a...,With much of the UK enduring another period of...
1,1,9267,'Liz Truss the Brief?' World reacts to UK poli...,"Mon, 17 Oct 2022 11:35:12 GMT",https://www.bbc.co.uk/news/world-63285480,https://www.bbc.co.uk/news/world-63285480?at_m...,The UK's political chaos has been watched arou...


In [157]:
titles = pandas.DataFrame(bbc_data['title']);
titles.head()

,title
0,Can I refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...
2,Rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...


CLEAN DATA (TOKENIZING)

In [158]:
titles['lowercase'] = titles['title'].str.lower()
titles.head()

,title,lowercase
0,Can I refuse to work?,can i refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...


In [159]:
en_stopwords = stopwords.words('english')
en_stopwords.remove('not')

titles['no_stopwords'] = titles['lowercase'].apply(lambda row : " ".join([word for word in row.split() if word not in en_stopwords]))

In [160]:
titles['no_stopwords'][:4]

0                                         refuse work?
1    'liz truss brief?' world reacts uk political t...
2      rationing energy nothing new off-grid community
3        hunt superyachts sanctioned russian oligarchs
Name: no_stopwords, dtype: object

In [161]:
# don't forget 'titles[no_stopwords] is a single column, meaning series, so if u try to do axis = 1, u get error
titles['no_sw_no_punct'] = titles['no_stopwords'].apply(lambda row: re.sub(r'([^\w\s])', '', row))

# this will work too, but here we need to put axis = 1 for row-wise check in dataframe 'titles'
#titles['no_sw_no_punct'] = titles.apply(lambda row: re.sub(r'([^\w\s])', '', row['no_stopwords']), axis = 1)

titles['no_sw_no_punct'].head()

0                                          refuse work
1    liz truss brief world reacts uk political turmoil
2       rationing energy nothing new offgrid community
3        hunt superyachts sanctioned russian oligarchs
4           platinum jubilee 70 years queen 70 seconds
Name: no_sw_no_punct, dtype: object

In [162]:
titles['tokens_rawWords'] = titles['title'].apply(lambda row : word_tokenize(row))
titles['tokens_cleanWords'] = titles['no_sw_no_punct'].apply(lambda row : word_tokenize(row))
# titles['tokens_rawWords'] #uncomment if you want to check


In [163]:
titles['tokens_cleanWords'].head(8)

0                                       [refuse, work]
1    [liz, truss, brief, world, reacts, uk, politic...
2    [rationing, energy, nothing, new, offgrid, com...
3    [hunt, superyachts, sanctioned, russian, oliga...
4    [platinum, jubilee, 70, years, queen, 70, seco...
5    [red, bull, found, guilty, breaking, formula, ...
6    [world, triathlon, championship, series, flora...
7    [terry, hall, coventry, scooter, rideout, pays...
Name: tokens_cleanWords, dtype: object

In [164]:
lemmatizer = WordNetLemmatizer()
titles['lemmatized_clean'] = titles['tokens_cleanWords'].apply(lambda tokens: [lemmatizer.lemmatize(token) for token in tokens])

In [165]:
titles['lemmatized_clean'].head()

0                                       [refuse, work]
1    [liz, truss, brief, world, reacts, uk, politic...
2    [rationing, energy, nothing, new, offgrid, com...
3    [hunt, superyachts, sanctioned, russian, oliga...
4     [platinum, jubilee, 70, year, queen, 70, second]
Name: lemmatized_clean, dtype: object

In [166]:
# creating one giant array of raw and clean words (all rows(and words in it) combined in one array)
tokens_raw_list = sum(titles['tokens_rawWords'], [])
tokens_clean_list = sum(titles['lemmatized_clean'], [])


In [167]:
tokens_raw_list[:10]

['Can', 'I', 'refuse', 'to', 'work', '?', "'Liz", 'Truss', 'the', 'Brief']

In [168]:
titles.head(2)

,title,lowercase,no_stopwords,no_sw_no_punct,tokens_rawWords,tokens_cleanWords,lemmatized_clean
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[Can, I, refuse, to, work, ?]","[refuse, work]","[refuse, work]"
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil,"['Liz, Truss, the, Brief, ?, ', World, reacts,...","[liz, truss, brief, world, reacts, uk, politic...","[liz, truss, brief, world, reacts, uk, politic..."


POS (Parts of Speech) TAGGING

In [169]:
nlp = spacy.load('en_core_web_sm')

#creating one giant string (like a big essay) of all tokens.
spacy_doc = nlp(' '.join(tokens_raw_list))


In [170]:
spacy_doc[:80]

Can I refuse to work ? 'Liz Truss the Brief ? ' World reacts to UK political turmoil Rationing energy is nothing new for off-grid community The hunt for superyachts of sanctioned Russian oligarchs Platinum Jubilee : 70 years of the Queen in 70 seconds Red Bull found guilty of breaking Formula 1 's budget cap World Triathlon Championship Series : Flora Duffy beats Georgia Taylor-Brown to women 's title Terry Hall : Coventry scooter

In [171]:
# creating dataframe with two columns token (for word), and pos_tag(what kind of word it is, e.g. Verb, noun)
pos_df = pandas.DataFrame(columns=['token', 'pos_tag'])

#add words in pos_df dataframe from spacy_doc

for token in spacy_doc:
    pos_df = pandas.concat([pos_df, pandas.DataFrame.from_records([{'token':token.text, 'pos_tag':token.pos_}])], ignore_index=True)

In [172]:
pos_df.head(10)

,token,pos_tag
0,Can,AUX
1,I,PRON
2,refuse,VERB
3,to,PART
4,work,VERB
5,?,PUNCT
6,',PUNCT
7,Liz,PROPN
8,Truss,PROPN
9,the,DET


In [173]:
pos_df_counts = pos_df.groupby(['token', 'pos_tag']).size().reset_index(name='counts').sort_values(by='counts', ascending=False)
pos_df_counts.head()


,token,pos_tag,counts
95,:,PUNCT,543
8,',PUNCT,300
2897,in,ADP,187
4082,to,PART,175
3268,of,ADP,172


In [174]:
#after grouping by 'pos_tag', we count the number of non-NaN rows in each group.
#it doesn't matter what u use after groupby()[here], it can also be 'pos_tag' again instead of 'token', result is same
pos_df_poscounts = pos_df_counts.groupby(['pos_tag'])['token'].count().sort_values(ascending=False)
pos_df_poscounts.head()

pos_tag
NOUN     1487
PROPN    1346
VERB      759
ADJ       390
NUM        87
Name: token, dtype: int64

In [175]:
nouns = pos_df_counts[pos_df_counts.pos_tag == 'NOUN'].reset_index(drop=True) #drop = True, will reset the row index(row number)
nouns[:8]

,token,pos_tag,counts
0,war,NOUN,35
1,record,NOUN,15
2,police,NOUN,14
3,year,NOUN,14
4,win,NOUN,14
5,living,NOUN,13
6,tax,NOUN,13
7,day,NOUN,12


POS - NER (Named Entity Recognition)

In [176]:
ner_df = pandas.DataFrame(columns=['token', 'ner_tag'])

for token in spacy_doc.ents:
    if pandas.isna(token.label_) is False: # isna() = "if the token is not is NaN, null or missing" is false, then process it
        ner_df = pandas.concat([ner_df, pandas.DataFrame.from_records([{'token':token.text, 'ner_tag':token.label_}])], ignore_index=True)

# in concat([df1, df2, df3, df4],more parameters), the first paramter is list because then it can concat multiple dfs in one go.

In [177]:
ner_df.head()

,token,ner_tag
0,Liz Truss,PERSON
1,UK,GPE
2,Rationing,PRODUCT
3,superyachts,CARDINAL
4,Russian,NORP


In [178]:
ner_df_counts = ner_df.groupby(['token','ner_tag']).size().reset_index(name='counts').sort_values(by='counts', ascending=False)

In [179]:
ner_df_counts.head()

,token,ner_tag,counts
965,Ukraine,GPE,47
955,UK,GPE,36
329,England,GPE,32
819,Russian,NORP,20
957,US,GPE,19


These techniquest above help us understand what kind of words appear in our data set, what they represent, and how they relate to the topic being discussed.